In [13]:
"""
TUS-GAN — ITUS 2019 Diary Encoding
====================================
Converts raw ITUS 2019 wide-format CSV into:
  • diary_tensor  : (N, 1, 48, 1)  float32  in [-1, +1]
  • cond_vector   : (N, C)         float32  (one-hot + embedding-ready)
  • district_ids  : (N,)           int64    (for the embedding layer)

Pipeline overview
-----------------
Step 1  Load & fix dtypes
Step 2  Drop / impute missing values
Step 3  Reorder time columns → 48 slots anchored at 04:00
Step 4  Normalise activity codes → [-1, +1]
Step 5  Build diary tensor  (N, 1, 48, 1)
Step 6  Encode conditioning features
         6a  Age → 7-class one-hot
         6b  Gender → one-hot (3)
         6c  Marital status → one-hot (4)
         6d  Education → one-hot (11)
         6e  Principal activity → one-hot (13)
         6f  Day of week → one-hot (7)
         6g  Sector → one-hot (2)
         6h  Caregiving → one-hot (2)
         6i  District → integer id  (fed to nn.Embedding later)
Step 7  Concatenate one-hot parts → final cond_vector
Step 8  Sanity checks & save
"""

import numpy as np
import pandas as pd

# ──────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────
CSV_PATH   = "../2019/TUS-2019-cleaned.csv"
SAVE_PATH  = "tusgan_encoded.npz"

# Activity code range in ITUS (1–99 typical).
# We map codes to [-1,+1] using  (code / MID) - 1
# where MID = (MAX_CODE + 1) / 2
MAX_ACTIVITY_CODE = 99
ACT_MID           = (MAX_ACTIVITY_CODE + 1) / 2


In [14]:

# ──────────────────────────────────────────────
# STEP 1 — LOAD & FIX DTYPES
# ──────────────────────────────────────────────
# Raw export often stores numeric columns as objects
# because of mixed NaN representations ("", " ", "nan").
# We coerce everything that should be numeric.

print("Step 1 — Loading CSV and fixing dtypes ...")

df = pd.read_csv(CSV_PATH, low_memory=False)

# Columns that must be numeric
NUMERIC_COLS = [
    "Person_Serial_No", "Gender", "Marital_Status",
    "Highest_level_of_education",
    "usual principal activity: status (code)",
    "age", "Serial no.of member",
    "unpaid/paid status of activity",
    "Sector", "District", "Household size",
    "Social group ",
    "usual monthly consumer expenditure E: [A+B+C+(D/12)]",
    "caregiving_dummy",
]

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Day of week: stored as float (1.0–7.0) — cast to nullable int
if "day of week" in df.columns:
    df["day of week"] = pd.to_numeric(df["day of week"], errors="coerce")

print(f"  Loaded {len(df):,} rows × {len(df.columns)} columns")


Step 1 — Loading CSV and fixing dtypes ...
  Loaded 4,472 rows × 66 columns


In [15]:
# ──────────────────────────────────────────────
# STEP 2 — DROP / IMPUTE MISSING VALUES
# ──────────────────────────────────────────────
# Strategy:
#   • Diary slots: forward-fill within a person's diary,
#     then fill any remaining NaN with 0 (rare boundary gaps).
#   • Conditioning features: rows with NaN in any conditioning
#     column are dropped — we cannot fabricate demographics.

print("\nStep 2 — Handling missing values ...")

# The 48 time-slot column names exactly as they appear in the CSV
# Order: 04:00→04:30, 04:30→05:00, ..., 03:30→04:00
TIME_COLS = [
    "04:00-04:30","04:30-05:00","05:00-05:30","05:30-06:00",
    "06:00-06:30","06:30-07:00","07:00-07:30","07:30-08:00",
    "08:00-08:30","08:30-09:00","09:00-09:30","09:30-10:00",
    "10:00-10:30","10:30-11:00","11:00-11:30","11:30-12:00",
    "12:00-12:30","12:30-13:00","13:00-13:30","13:30-14:00",
    "14:00-14:30","14:30-15:00","15:00-15:30","15:30-16:00",
    "16:00-16:30","16:30-17:00","17:00-17:30","17:30-18:00",
    "18:00-18:30","18:30-19:00","19:00-19:30","19:30-20:00",
    "20:00-20:30","20:30-21:00","21:00-21:30","21:30-22:00",
    "22:00-22:30","22:30-23:00","23:00-23:30","23:30-00:00",
    "00:00-00:30","00:30-01:00","01:00-01:30","01:30-02:00",
    "02:00-02:30","02:30-03:00","03:00-03:30","03:30-04:00",
]
assert len(TIME_COLS) == 48, "Expected exactly 48 time columns"

# Forward-fill diary NaNs (same activity carries forward)
df[TIME_COLS] = df[TIME_COLS].ffill(axis=1).fillna(0)

# Drop rows where any conditioning feature is missing
COND_REQUIRED = [
    "Gender", "Marital_Status", "Highest_level_of_education",
    "usual principal activity: status (code)",
    "day of week", "age", "Sector", "District",
]
before = len(df)
df = df.dropna(subset=COND_REQUIRED)
print(f"  Dropped {before - len(df):,} rows with missing conditioning values")
print(f"  Remaining: {len(df):,} rows")



Step 2 — Handling missing values ...
  Dropped 16 rows with missing conditioning values
  Remaining: 4,456 rows


In [16]:
# ──────────────────────────────────────────────
# STEP 3 — REORDER TIME COLUMNS (already in order)
# ──────────────────────────────────────────────
# ITUS 2019 wide format starts at 04:00 — our anchor.
# TIME_COLS is already ordered 04:00 → 03:30, so no reordering needed.
# We extract the 48-slot matrix directly.

print("\nStep 3 — Extracting 48-slot diary matrix ...")

diary_raw = df[TIME_COLS].values.astype(np.float32)   # (N, 48)
print(f"  diary_raw shape : {diary_raw.shape}")
print(f"  activity code range in data: {diary_raw.min():.0f} – {diary_raw.max():.0f}")



Step 3 — Extracting 48-slot diary matrix ...
  diary_raw shape : (4456, 48)
  activity code range in data: 0 – 7


In [17]:
# ──────────────────────────────────────────────
# STEP 4 — NORMALISE ACTIVITY CODES → [-1, +1]
# ──────────────────────────────────────────────
# Formula:  norm = (code / ACT_MID) - 1
#
# code=1  → (1/50) - 1 = -0.98   (just below -1, clipped)
# code=50 → (50/50) - 1 =  0.00
# code=99 → (99/50) - 1 = +0.98
#
# We clip to [-1, +1] so the critic and generator are always
# operating on the same exact range as Tanh output.

print("\nStep 4 — Normalising activity codes to [-1, +1] ...")

diary_norm = (diary_raw / ACT_MID) - 1.0
diary_norm = np.clip(diary_norm, -1.0, 1.0)

print(f"  Normalised range: {diary_norm.min():.4f} – {diary_norm.max():.4f}")



Step 4 — Normalising activity codes to [-1, +1] ...
  Normalised range: -1.0000 – -0.8600


In [18]:

# ──────────────────────────────────────────────
# STEP 5 — BUILD DIARY TENSOR  (N, 1, 48, 1)
# ──────────────────────────────────────────────
# Shape explanation:
#   N  = number of person-days (one row = one diary)
#   1  = single channel (activity code only)
#   48 = time slots
#   1  = singleton spatial dim for Conv compatibility

print("\nStep 5 — Reshaping into diary tensor (N, 1, 48, 1) ...")

N = len(diary_norm)
diary_tensor = diary_norm.reshape(N, 1, 48, 1)

print(f"  diary_tensor shape : {diary_tensor.shape}")
print(f"  dtype              : {diary_tensor.dtype}")



Step 5 — Reshaping into diary tensor (N, 1, 48, 1) ...
  diary_tensor shape : (4456, 1, 48, 1)
  dtype              : float32


In [19]:
# ──────────────────────────────────────────────
# STEP 6 — ENCODE CONDITIONING FEATURES
# ──────────────────────────────────────────────

print("\nStep 6 — Encoding conditioning features ...")

# ── 6a  AGE → 7-class one-hot ──────────────────
# Bin edges (right-exclusive): <15, 15-17, 18-24, 25-34, 35-44, 45-59, 60+
# np.digitize(x, bins) returns index of the bin:
#   x < 15   → 0   "Childhood & Middle School"
#   15 ≤ x < 18 → 1   "School Students"
#   18 ≤ x < 25 → 2   "College / Early Work"
#   25 ≤ x < 35 → 3   "Early Career / New Parenthood"
#   35 ≤ x < 45 → 4   "Mid-Career / Older Children"
#   45 ≤ x < 60 → 5   "Later Working Years"
#   x ≥ 60      → 6   "Retirement"

AGE_BINS      = [15, 18, 25, 35, 45, 60]          # 6 edges → 7 buckets
AGE_LABELS    = [
    "Childhood & Middle School",   # 0
    "School Students",             # 1
    "College / Early Work",        # 2
    "Early Career / New Parenthood", # 3
    "Mid-Career / Older Children", # 4
    "Later Working Years",         # 5
    "Retirement",                  # 6
]

age_vals  = df["age"].values.astype(np.float32)
age_codes = np.digitize(age_vals, AGE_BINS)        # values 0–6
age_oh    = np.eye(7, dtype=np.float32)[age_codes] # (N, 7)

print(f"  6a  age_onehot    : {age_oh.shape}  | classes: {np.unique(age_codes)}")


# ── 6b  GENDER → one-hot (3) ───────────────────
# Values in data: 1=Male, 2=Female, 3=Transgender
# Remap to 0-indexed: 1→0, 2→1, 3→2

gender_vals  = df["Gender"].values.astype(np.int32) - 1   # 0,1,2
gender_vals  = np.clip(gender_vals, 0, 2)                  # safety clip
gender_oh    = np.eye(3, dtype=np.float32)[gender_vals]    # (N, 3)

print(f"  6b  gender_onehot : {gender_oh.shape}  | values seen: {np.unique(gender_vals)}")


# ── 6c  MARITAL STATUS → one-hot (4) ───────────
# Values: 1,2,3,4 → remap to 0-indexed

marital_vals = df["Marital_Status"].values.astype(np.int32) - 1
marital_vals = np.clip(marital_vals, 0, 3)
marital_oh   = np.eye(4, dtype=np.float32)[marital_vals]   # (N, 4)

print(f"  6c  marital_onehot: {marital_oh.shape}  | values seen: {np.unique(marital_vals)}")


# ── 6d  EDUCATION → one-hot (11) ───────────────
# Values: 1,2,3,4,5,6,7,8,10,11,12
# We map them to 0-indexed positions using a lookup dict.
# (They are NOT sequential, so we cannot simply subtract 1.)

EDU_CODES  = [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12]
edu_to_idx = {v: i for i, v in enumerate(EDU_CODES)}       # {1:0, 2:1, ..., 12:10}

edu_raw    = df["Highest_level_of_education"].values.astype(np.int32)
edu_idx    = np.array([edu_to_idx.get(v, 0) for v in edu_raw], dtype=np.int32)
edu_oh     = np.eye(11, dtype=np.float32)[edu_idx]          # (N, 11)

print(f"  6d  edu_onehot    : {edu_oh.shape}  | unique idx: {np.unique(edu_idx)}")


# ── 6e  PRINCIPAL ACTIVITY → one-hot (13) ──────
# Values: 11,12,21,31,41,51,81,91,92,93,94,95,97
# NaN rows were already dropped in Step 2.

ACT_CODES   = [11, 12, 21, 31, 41, 51, 81, 91, 92, 93, 94, 95, 97]
act_to_idx  = {v: i for i, v in enumerate(ACT_CODES)}

act_raw     = df["usual principal activity: status (code)"].values.astype(np.int32)
act_idx     = np.array([act_to_idx.get(v, 0) for v in act_raw], dtype=np.int32)
act_oh      = np.eye(13, dtype=np.float32)[act_idx]         # (N, 13)

print(f"  6e  act_onehot    : {act_oh.shape}  | unique idx: {np.unique(act_idx)}")


# ── 6f  DAY OF WEEK → one-hot (7) ──────────────
# Values: 1–7  (Monday=1 … Sunday=7 in ITUS)
# Remap to 0-indexed: 1→0, ..., 7→6

dow_vals    = df["day of week"].values.astype(np.int32) - 1
dow_vals    = np.clip(dow_vals, 0, 6)
dow_oh      = np.eye(7, dtype=np.float32)[dow_vals]         # (N, 7)

print(f"  6f  dow_onehot    : {dow_oh.shape}  | values seen: {np.unique(dow_vals)}")


# ── 6g  SECTOR → one-hot (2) ───────────────────
# Values: 1=Rural, 2=Urban → remap 0-indexed

sector_vals = df["Sector"].values.astype(np.int32) - 1
sector_vals = np.clip(sector_vals, 0, 1)
sector_oh   = np.eye(2, dtype=np.float32)[sector_vals]      # (N, 2)

print(f"  6g  sector_onehot : {sector_oh.shape}  | values seen: {np.unique(sector_vals)}")


# ── 6i  DISTRICT → integer id (for nn.Embedding) ──
# 71 unique district codes — high cardinality, so we use a
# learned embedding rather than one-hot.
# Here we just build the integer id array.
# In your model:  nn.Embedding(num_districts, embed_dim=16)

all_districts = sorted(df["District"].dropna().unique().astype(int).tolist())
dist_to_idx   = {v: i for i, v in enumerate(all_districts)}
NUM_DISTRICTS = len(all_districts)

district_ids  = np.array(
    [dist_to_idx.get(int(v), 0) for v in df["District"].values],
    dtype=np.int64
)

print(f"  6i  district_ids  : {district_ids.shape}  | {NUM_DISTRICTS} unique districts")



# ── 6h  CAREGIVING → one-hot (2) ────────────────
# Values in data: 1.0=Yes, 0.0=No, NaN=No
care_vals = df["caregiving_dummy"].fillna(0).values.astype(np.int32)
care_vals = np.clip(care_vals, 0, 1)
care_oh   = np.eye(2, dtype=np.float32)[care_vals]
print(f"  6h  care_onehot   : {care_oh.shape}  | values seen: {np.unique(care_vals)}")



Step 6 — Encoding conditioning features ...
  6a  age_onehot    : (4456, 7)  | classes: [0 1 2 3 4 5 6]
  6b  gender_onehot : (4456, 3)  | values seen: [0 1 2]
  6c  marital_onehot: (4456, 4)  | values seen: [0 1 2 3]
  6d  edu_onehot    : (4456, 11)  | unique idx: [ 0  1  2  3  4  5  6  7  8  9 10]
  6e  act_onehot    : (4456, 13)  | unique idx: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
  6f  dow_onehot    : (4456, 7)  | values seen: [0 1 2 3 4 5 6]
  6g  sector_onehot : (4456, 2)  | values seen: [0 1]
  6h  district_ids  : (4456,)  | 71 unique districts


In [20]:

# ──────────────────────────────────────────────
# STEP 7 — CONCATENATE ONE-HOT PARTS → cond_vector
# ──────────────────────────────────────────────
# Breakdown:
#   age      (7) + gender   (3) + marital (4) +
#   edu     (11) + activity(13) + dow     (7) +
#   sector   (2) + caregiving (2)
#   ─────────────────────────────────────────
#   total   49 dims  (+ 1 district embedding added in model)

print("\nStep 7 — Concatenating conditioning vector ...")

cond_vector = np.concatenate([
    age_oh,        # (N, 7)
    gender_oh,     # (N, 3)
    marital_oh,    # (N, 4)
    edu_oh,        # (N, 11)
    act_oh,        # (N, 13)
    dow_oh,        # (N, 7)
    sector_oh,     # (N, 2)
    care_oh,       # (N, 2)
], axis=1)         # → (N, 49)

print(f"  cond_vector shape : {cond_vector.shape}")
print(f"  district_ids shape: {district_ids.shape}  (separate, fed to Embedding)")



Step 7 — Concatenating conditioning vector ...
  cond_vector shape : (4456, 47)
  district_ids shape: (4456,)  (separate, fed to Embedding)


In [21]:

# ──────────────────────────────────────────────
# STEP 8 — SANITY CHECKS & SAVE
# ──────────────────────────────────────────────

print("\nStep 8 — Sanity checks ...")

# 1. Tensor shape
assert diary_tensor.shape == (N, 1, 48, 1), \
    f"Diary tensor shape mismatch: {diary_tensor.shape}"

# 2. Value range
assert diary_tensor.min() >= -1.0 and diary_tensor.max() <= 1.0, \
    "Diary values outside [-1, +1]"

# 3. Condition vector shape
assert cond_vector.shape == (N, 49), \
    f"Cond vector shape mismatch: {cond_vector.shape}"

# 4. One-hot rows sum to 1 for all parts
assert np.allclose(age_oh.sum(axis=1), 1), "age_oh rows don't sum to 1"
assert np.allclose(gender_oh.sum(axis=1), 1), "gender_oh rows don't sum to 1"
assert np.allclose(marital_oh.sum(axis=1), 1), "marital_oh rows don't sum to 1"
assert np.allclose(edu_oh.sum(axis=1), 1), "edu_oh rows don't sum to 1"
assert np.allclose(act_oh.sum(axis=1), 1), "act_oh rows don't sum to 1"
assert np.allclose(dow_oh.sum(axis=1), 1), "dow_oh rows don't sum to 1"
assert np.allclose(sector_oh.sum(axis=1), 1), "sector_oh rows don't sum to 1"

print("  All checks passed ✓")

# Save
np.savez_compressed(
    SAVE_PATH,
    diary_tensor  = diary_tensor,    # (N, 1, 48, 1)  float32
    cond_vector   = cond_vector,     # (N, 49)         float32
    district_ids  = district_ids,    # (N,)            int64
    num_districts = np.array(NUM_DISTRICTS),
)

print(f"\n  Saved → {SAVE_PATH}")
print(f"  diary_tensor  : {diary_tensor.shape}  float32")
print(f"  cond_vector   : {cond_vector.shape}  float32")
print(f"  district_ids  : {district_ids.shape}   int64")
print(f"  num_districts : {NUM_DISTRICTS}")


)assert np.allclose(care_oh.sum(axis=1), 1), "care_oh rows don't sum to 1"



Step 8 — Sanity checks ...
  All checks passed ✓

  Saved → tusgan_encoded.npz
  diary_tensor  : (4456, 1, 48, 1)  float32
  cond_vector   : (4456, 47)  float32
  district_ids  : (4456,)   int64
  num_districts : 71


'\nimport torch\nimport torch.nn as nn\nimport numpy as np\n\ndata = np.load("tusgan_encoded.npz")\n\ndiary      = torch.from_numpy(data["diary_tensor"])   # (N, 1, 48, 1)\ncond       = torch.from_numpy(data["cond_vector"])    # (N, 47)\ndist_ids   = torch.from_numpy(data["district_ids"])   # (N,)\nN_DIST     = int(data["num_districts"])\n\n# District embedding layer (inside your Generator / Critic)\ndistrict_emb = nn.Embedding(N_DIST, embedding_dim=16)\n\n# At training time, merge district embedding with cond vector:\n#   dist_vec  = district_emb(dist_ids_batch)          # (B, 16)\n#   full_cond = torch.cat([cond_batch, dist_vec], 1)  # (B, 47+16=63)\n\n# Generator receives:  torch.cat([z, full_cond], dim=1)\n#   z shape      : (B, 128)\n#   full_cond    : (B, 63)\n#   gen input    : (B, 191)\n'

In [22]:
# ──────────────────────────────────────────────
# HOW TO USE IN YOUR PYTORCH DATASET
# ──────────────────────────────────────────────

import torch
import torch.nn as nn
import numpy as np

data = np.load("tusgan_encoded.npz")

diary      = torch.from_numpy(data["diary_tensor"])   # (N, 1, 48, 1)
cond       = torch.from_numpy(data["cond_vector"])    # (N, 49)
dist_ids   = torch.from_numpy(data["district_ids"])   # (N,)
N_DIST     = int(data["num_districts"])

# District embedding layer (inside your Generator / Critic)
district_emb = nn.Embedding(N_DIST, embedding_dim=16)

# At training time, merge district embedding with cond vector:
#   dist_vec  = district_emb(dist_ids_batch)          # (B, 16)
#   full_cond = torch.cat([cond_batch, dist_vec], 1)  # (B, 49+16=65)

# Generator receives:  torch.cat([z, full_cond], dim=1)
#   z shape      : (B, 128)
#   full_cond    : (B, 65)
#   gen input    : (B, 193

In [25]:
diary[0]

tensor([[[-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-0.9200],
         [-0.9200],
         [-0.9600],
         [-0.9600],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-0.9600],
         [-0.9600],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-0.9200],
         [-0.9200],
         [-0.9200],
         [-0.9200],
         [-0.9200],
         [-0.9200],
         [-0.9200],
         [-0.9200],
         [-0.9600],
         [-0.9600],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000],
         [-1.0000]]])